# Feature Engineering: Sleep Features

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path
notebook_dir = Path().resolve()
sys.path.append(str(notebook_dir.parent))        # ../
sys.path.append(str(notebook_dir.parent.parent)) # ../../


import pandas as pd
from datetime import datetime
from plotly.subplots import make_subplots
import plotly.graph_objects as go

from data_helper.clean_data_functions import clean_data_baseline_optimized, clean_data_no_data_days_optimized
from data_helper.download_REDCap_data import download_files_for_records
from plot_helper.descriptive_stats_plot import plot_descriptive_stats
from plot_helper.colors import COLORS

#dataframe for all features
df_features = pd.DataFrame()

## Download Data from REDCap
(Comment out if not needed)


In [ ]:
download_files_for_records(['all'], "venu3_sleep_summary", "baseline_period_arm_1", "data/sleep_summary")

download_files_for_records(['all'], "venu3_sleep_stage", "baseline_period_arm_1", "data/sleep_stage")

## Create Base DataFrames

In [ ]:
#Get downloaded data
folder_sleep_summary = Path("../../data/sleep_summary")
folder_sleep_stage = Path("../../data/sleep_stage")

#Get merged csv file paths
sleep_summary_files = [f.path for f in os.scandir(folder_sleep_summary) if f.is_file() and f.name.endswith(".csv")]
sleep_stage_files = [f.path for f in os.scandir(folder_sleep_stage) if f.is_file() and f.name.endswith(".csv")]

#create dataframes
df_sleep_summary_raw = pd.concat(
    [pd.read_csv(f) for f in sleep_summary_files],
    ignore_index=True
)
df_sleep_summary_raw['durationInHrs'] = df_sleep_summary_raw['durationInMs'] / (1000 * 60 * 60)

df_sleep_stage_raw = pd.concat(
    [pd.read_csv(f) for f in sleep_stage_files],
    ignore_index=True
)
df_sleep_stage_raw.head()

#clean dataframes
##match baseline dates
print(f"Sleep summary data before first cropping: {df_sleep_summary_raw.shape}")
print(f"Sleep stage data before first cropping: {df_sleep_stage_raw.shape}")
df_sleep_summary_raw, df_sleep_stage_raw = clean_data_baseline_optimized([df_sleep_summary_raw, df_sleep_stage_raw])
print(f"Sleep summary data after first cropping: {df_sleep_summary_raw.shape}")
print(f"Sleep stage data after first cropping: {df_sleep_stage_raw.shape}")
print(f"------"*20)


#*--Make sure sleep does not start too early as calendarDate == wake up date
#*drop rows where calendarDate is smaller than start visit date plus 1 day or calendarDate is larger than end visit date
df_visit_dates = pd.read_csv("../../data/checks/visit_dates_2026-07-08.csv")
df_visit_dates_v1 = df_visit_dates[df_visit_dates['visit_name'] == 'V1'].copy()

# Convert dates
df_sleep_summary_raw["calendarDate"] = pd.to_datetime(
    df_sleep_summary_raw["calendarDate"],
    errors="coerce"
)

df_visit_dates_v1["start_visit_date"] = pd.to_datetime(
    df_visit_dates_v1["start_visit_date"],
    errors="coerce"
)

df_visit_dates_v1["end_visit_date"] = pd.to_datetime(
    df_visit_dates_v1["end_visit_date"],
    errors="coerce"
)
# Keep only needed visit columns
df_visit_dates_v1 = df_visit_dates_v1[
    ["study_id", "start_visit_date", "end_visit_date"]
].drop_duplicates("study_id")

# Merge visit dates onto sleep summary
df_sleep_summary_raw = df_sleep_summary_raw.merge(
    df_visit_dates_v1,
    on="study_id",
    how="left"
)

start_cutoff = df_sleep_summary_raw["start_visit_date"] + pd.Timedelta(days=1)

mask_keep = (
    (df_sleep_summary_raw["calendarDate"] >= start_cutoff) &
    (df_sleep_summary_raw["calendarDate"] <= df_sleep_summary_raw["end_visit_date"])
)
df_sleep_summary_raw = df_sleep_summary_raw[mask_keep].copy()

print(f"Sleep summary data after second cropping: {df_sleep_summary_raw.shape}")
display(df_sleep_summary_raw.head())
print("------"*20)


##drop days with no data
df_sleep_summary_raw, df_sleep_stage_raw = clean_data_no_data_days_optimized([df_sleep_summary_raw, df_sleep_stage_raw])
print(f"Sleep summary data after dropping days with no data: {df_sleep_summary_raw.shape}")
display(df_sleep_summary_raw.head())
print(f"Sleep stage data after dropping days with no data: {df_sleep_stage_raw.shape}")
display(df_sleep_stage_raw.head())
print("------"*20)

#get number of patients
n_sleep_summary_raw = df_sleep_summary_raw['study_id'].nunique()
print(f"Number of patients in sleep summary data: {n_sleep_summary_raw}")
#get number of patients
n_sleep_stage_raw = df_sleep_stage_raw['study_id'].nunique()
print(f"Number of patients in sleep stage data: {n_sleep_stage_raw}")

#get study ids ordered in ascending order
study_ids_sleep_summary = sorted(df_sleep_summary_raw['study_id'].unique())
print(f"Study IDs in sleep summary data: {study_ids_sleep_summary}")

study_ids_sleep_stage = sorted(df_sleep_stage_raw['study_id'].unique())
print(f"Study IDs in sleep stage data: {study_ids_sleep_stage}")


## Total Sleep Duration

In [ ]:
df_sleep_f1 = df_sleep_summary_raw[['study_id', 'calendarDate', 'datetime', 'durationInMs', 'durationInHrs']].copy()
df_sleep_f1.head()

df_sleep_f1 = df_sleep_f1.sort_values(by=['study_id']).reset_index(drop=True)

#calculate descriptive stats for each study id
total_sleep_duration = df_sleep_f1.groupby('study_id')['durationInHrs'].agg(
    mean_sleep_duration='mean',
    median_sleep_duration='median',
    std_sleep_duration='std',
    skewness_sleep_duration=('skew')  # positive = right skew
).reset_index()

display(total_sleep_duration.head())


#create plots
fig_f1 = plot_descriptive_stats(
    df = df_sleep_f1,
    col_df = "durationInHrs", 
    df_descriptive_stats=total_sleep_duration,
    col_mean="mean_sleep_duration",
    col_median="median_sleep_duration", 
    n_patients=n_sleep_summary_raw,
    feature="Sleep Duration",
    title="Sleep Duration [h]",
    tickformat=".2f",
    colors = COLORS
    )

fig_f1.show()


#add sleep duration stats to feature dataframe
df_features = total_sleep_duration[['study_id', 'mean_sleep_duration', 'median_sleep_duration', 'std_sleep_duration']].copy()

#create daily dataframe
df_features_daily  = df_sleep_f1[['study_id', 'calendarDate', 'durationInHrs']].copy()
df_features_daily = df_features_daily.rename(columns={'durationInHrs': 'daily_sleep_duration_hrs'})
display(df_features_daily.head())

## Onset Time

In [ ]:
df_sleep_f2 = df_sleep_summary_raw[['study_id', 'calendarDate', 'datetime', 'datetime_utc', 'timezone']].copy()

dt_utc = pd.to_datetime(
            df_sleep_f2["datetime_utc"],
            errors="coerce",
            utc=True
        )
df_sleep_f2['sleep_onset'] = [
            ts.tz_convert(tz).time() if pd.notna(ts) and pd.notna(tz) else pd.NaT
            for ts, tz in zip(dt_utc, df_sleep_f2["timezone"])
        ]
display(df_sleep_f2.head())

df_sleep_f2 = df_sleep_f2.sort_values(by=['study_id']).reset_index(drop=True)
display(df_sleep_f2.sort_values(by=['sleep_onset'], ascending=False).head())
# Convert sleep_onset time to seconds since midnight for calculation
df_sleep_f2['onset_seconds'] = df_sleep_f2['sleep_onset'].apply(
    lambda x: x.hour * 3600 + x.minute * 60 + x.second
)

# Handle midnight crossover (adjust times before midday to be "past midnight")
df_sleep_f2['onset_seconds_adjusted'] = df_sleep_f2['onset_seconds'].apply(
    lambda x: x + 24 * 3600 if x < 12 * 3600 else x
)
# Keep as datetime for plotting (plotly handles datetime axes natively)
df_sleep_f2['onset_plot'] = df_sleep_f2['onset_seconds_adjusted'].apply(
    lambda x: pd.Timestamp('2000-01-01') + pd.Timedelta(seconds=x)
)

# Calculate mean, std and median per study_id
sleep_onset = df_sleep_f2.groupby('study_id')['onset_seconds_adjusted'].agg(
    mean_onset='mean',
    median_onset='median',
    std_onset='std',
).reset_index()


# Convert back to readable time
sleep_onset['mean_sleep_onset'] = sleep_onset['mean_onset'].apply(
    lambda x: pd.Timestamp('00:00:00') + pd.Timedelta(seconds=x % (24 * 3600))
).dt.time

sleep_onset['median_sleep_onset'] = sleep_onset['median_onset'].apply(
    lambda x: pd.Timestamp('00:00:00') + pd.Timedelta(seconds=x % (24 * 3600))
).dt.time# Convert back to readable time
sleep_onset['std_sleep_onset_minutes'] = sleep_onset['std_onset'] / 60  
#for plotting
sleep_onset['mean_onset_plot'] = sleep_onset['mean_onset'].apply(
    lambda x: pd.Timestamp('2000-01-01') + pd.Timedelta(seconds=x)
)
sleep_onset['median_onset_plot'] = sleep_onset['median_onset'].apply(
    lambda x: pd.Timestamp('2000-01-01') + pd.Timedelta(seconds=x)
)

display(sleep_onset.head())

#create plots
fig_f2 = plot_descriptive_stats(
    df = df_sleep_f2,
    col_df = "onset_plot", 
    df_descriptive_stats=sleep_onset,
    col_mean="mean_onset_plot",
    col_median="median_onset_plot", 
    n_patients=n_sleep_summary_raw,
    feature="Sleep Onset",
    title="Sleep Onset Time [hh:mm]", 
    tickformat="%H:%M",
    bins = dict(size=60*60*1000),
    bins_stats = dict(size=60*60*1000),
    colors = COLORS
        )
fig_f2.show()


#add onset stats to feature dataframe
df_features = df_features.merge(
    sleep_onset[['study_id', 'mean_sleep_onset', 'median_sleep_onset', 'std_sleep_onset_minutes']],
    on='study_id',
    how='outer'
)
display(df_features.head())

#add daily onset to df_features_daily
df_features_daily = df_features_daily.merge(
    df_sleep_f2[['study_id', 'calendarDate', 'sleep_onset', 'onset_seconds_adjusted']],
    on=['study_id', 'calendarDate'],
    how='outer'
)
df_features_daily = df_features_daily.rename(columns={'sleep_onset': 'daily_sleep_onset'})
display(df_features_daily.head())

## Wake Up Time

In [ ]:
df_sleep_f3 = df_sleep_summary_raw[['study_id', 'calendarDate', 'datetime', 'datetime_utc', 'timezone', 'durationInMs', 'durationInHrs']].copy()

dt_utc = pd.to_datetime(
            df_sleep_f3["datetime_utc"],
            errors="coerce",
            utc=True
        )
df_sleep_f3['sleep_onset'] = [
            ts.tz_convert(tz) if pd.notna(ts) and pd.notna(tz) else pd.NaT
            for ts, tz in zip(dt_utc, df_sleep_f3["timezone"])
        ]

#wakeup_time = datetime.time + durationInMs
df_sleep_f3['wakeup_time'] = (df_sleep_f3['sleep_onset'] + pd.to_timedelta(df_sleep_f3['durationInMs'], unit='ms')).dt.time

df_sleep_f3 = df_sleep_f3.sort_values(by=['study_id']).reset_index(drop=True)

display(df_sleep_f3.head())

# Convert wakeup_time to plottable timestamp
df_sleep_f3['wakeup_seconds'] = df_sleep_f3['wakeup_time'].apply(
    lambda x: x.hour * 3600 + x.minute * 60 + x.second
)

# No midnight crossover needed here - all wakeups are in early morning
df_sleep_f3['wakeup_plot'] = df_sleep_f3['wakeup_seconds'].apply(
    lambda x: pd.Timestamp('2000-01-01') + pd.Timedelta(seconds=x)
)

# Calculate mean per study_id
wakeup_stats = df_sleep_f3.groupby('study_id')['wakeup_seconds'].agg(
    mean_wakeup='mean',
    median_wakeup='median',
    std_wakeup='std'
).reset_index()

wakeup_stats['mean_wakeup_plot'] = wakeup_stats['mean_wakeup'].apply(
    lambda x: pd.Timestamp('2000-01-01') + pd.Timedelta(seconds=x)
)
wakeup_stats['median_wakeup_plot'] = wakeup_stats['median_wakeup'].apply(
    lambda x: pd.Timestamp('2000-01-01') + pd.Timedelta(seconds=x)
)
#convert to readable time
wakeup_stats['mean_wakeup'] = wakeup_stats['mean_wakeup'].apply(
    lambda x: pd.Timestamp('00:00:00') + pd.Timedelta(seconds=x % (24 * 3600))
).dt.time
wakeup_stats['median_wakeup'] = wakeup_stats['median_wakeup'].apply(
    lambda x: pd.Timestamp('00:00:00') + pd.Timedelta(seconds=x % (24 * 3600))
).dt.time
wakeup_stats['std_wakeup'] = wakeup_stats['std_wakeup'].apply(
    lambda x: (pd.Timestamp('00:00:00') + pd.Timedelta(seconds=x % (24 * 3600))).time()
    if pd.notna(x) else pd.NaT
)

display(wakeup_stats.head())

#create plots
fig_f3 = plot_descriptive_stats(
    df = df_sleep_f3,
    col_df = "wakeup_plot", 
    df_descriptive_stats=wakeup_stats,
    col_mean="mean_wakeup_plot",
    col_median="median_wakeup_plot", 
    n_patients=n_sleep_summary_raw,
    feature="Wakeup Times",
    title="Wakeup Time [hh:mm]", 
    tickformat="%H:%M",
    bins = dict(size=60*60*1000),
    bins_stats = dict(size=60*60*1000),
    colors = COLORS
    )
fig_f3.show()


#add wakeup stats to feature dataframe
df_features = df_features.merge(
    wakeup_stats[['study_id', 'mean_wakeup', 'median_wakeup', 'std_wakeup']],
    on='study_id',
    how='outer'
)
display(df_features.head())

#add daily wakeup to df_features_daily
df_features_daily = df_features_daily.merge(
    df_sleep_f3[['study_id', 'calendarDate', 'wakeup_time', 'wakeup_seconds']],
    on=['study_id', 'calendarDate'],
    how='outer'
)
df_features_daily = df_features_daily.rename(columns={'wakeup_time': 'daily_wakeup_time '})
display(df_features_daily.head())

## Midpoint of Sleep

In [ ]:
df_sleep_f4 = df_sleep_summary_raw[['study_id', 'calendarDate', 'datetime', 'datetime_utc', 'timezone', 'durationInMs', 'durationInHrs']].copy()

dt_utc = pd.to_datetime(
            df_sleep_f4["datetime_utc"],
            errors="coerce",
            utc=True
        )
df_sleep_f4['sleep_onset'] = [
            ts.tz_convert(tz) if pd.notna(ts) and pd.notna(tz) else pd.NaT
            for ts, tz in zip(dt_utc, df_sleep_f4["timezone"])
        ]

#midpoint_time = datetime.time + durationInMs/2
df_sleep_f4['midpoint_time'] = (df_sleep_f4['sleep_onset'] + pd.to_timedelta(df_sleep_f4['durationInMs']/2, unit='ms')).dt.time

df_sleep_f4 = df_sleep_f4.sort_values(by=['study_id']).reset_index(drop=True)

display(df_sleep_f4.head())

# Convert midpoint_time to seconds
df_sleep_f4['midpoint_seconds'] = df_sleep_f4['midpoint_time'].apply(
    lambda x: x.hour * 3600 + x.minute * 60 + x.second
)

# Midnight crossover - threshold at 12:00 (midpoints before noon are past midnight)
df_sleep_f4['midpoint_seconds_adjusted'] = df_sleep_f4['midpoint_seconds'].apply(
    lambda x: x + 24 * 3600 if x < 12 * 3600 else x
)

# Convert to plottable timestamp
df_sleep_f4['midpoint_plot'] = df_sleep_f4['midpoint_seconds_adjusted'].apply(
    lambda x: pd.Timestamp('2000-01-01') + pd.Timedelta(seconds=x)
)
display(df_sleep_f4.head())
# Calculate mean per study_id
midpoint_stats = df_sleep_f4.groupby('study_id')['midpoint_seconds_adjusted'].agg(
    mean_midpoint_adj='mean',
    median_midpoint_adj='median',
    std_midpoint_adj='std'
).reset_index()

midpoint_stats['mean_midpoint_plot'] = midpoint_stats['mean_midpoint_adj'].apply(
    lambda x: pd.Timestamp('2000-01-01') + pd.Timedelta(seconds=x)
)
midpoint_stats['median_midpoint_plot'] = midpoint_stats['median_midpoint_adj'].apply(
    lambda x: pd.Timestamp('2000-01-01') + pd.Timedelta(seconds=x)
)
# Convert back to readable time
midpoint_stats['mean_midpoint'] = midpoint_stats['mean_midpoint_adj'].apply(
    lambda x: pd.Timestamp('00:00:00') + pd.Timedelta(seconds=x )
).dt.time
midpoint_stats['median_midpoint'] = midpoint_stats['median_midpoint_adj'].apply(
    lambda x: pd.Timestamp('00:00:00') + pd.Timedelta(seconds=x )
).dt.time
midpoint_stats['std_midpoint'] = midpoint_stats['std_midpoint_adj'].apply(
    lambda x: (pd.Timestamp('00:00:00') + pd.Timedelta(seconds=x)).time()
    if pd.notna(x) else pd.NaT
)

display(midpoint_stats.head())


#create plots
fig_f4 = plot_descriptive_stats(
    df = df_sleep_f4,
    col_df = "midpoint_plot", 
    df_descriptive_stats=midpoint_stats,
    col_mean="mean_midpoint_plot",
    col_median="median_midpoint_plot", 
    n_patients=n_sleep_summary_raw,
    feature="Sleep Midpoint",
    title="Sleep Midpoint Time [hh:mm]", 
    tickformat="%H:%M",
    bins = dict(size=60*60*1000),
    bins_stats = dict(size=60*60*1000),
    colors = COLORS
    )
fig_f4.show()


#add midpoint stats to feature dataframe
df_features = df_features.merge(
    midpoint_stats[['study_id', 'mean_midpoint', 'median_midpoint', 'std_midpoint']],
    on='study_id',
    how='outer'
)
display(df_features.head())

#add daily midpoint to df_features_daily
df_features_daily = df_features_daily.merge(
    df_sleep_f4[['study_id', 'calendarDate', 'midpoint_time', 'midpoint_seconds_adjusted']],
    on=['study_id', 'calendarDate'],
    how='outer'
)
df_features_daily = df_features_daily.rename(columns={'midpoint_time': 'daily_midpoint_time'})
display(df_features_daily.head())

## Wake-up After Sleep Onset WASO

### Duration

In [ ]:
df_sleep_f5 = df_sleep_summary_raw[['study_id', 'calendarDate', 'datetime', 'awakeDurationInMs']].copy()

df_sleep_f5['awakeDurationInHrs'] = df_sleep_f5['awakeDurationInMs'] / (1000 * 60 * 60)
df_sleep_f5['awakeDurationInMin'] = df_sleep_f5['awakeDurationInMs'] / (1000 * 60)

df_sleep_f5 = df_sleep_f5.sort_values(by=['study_id']).reset_index(drop=True)

display(df_sleep_f5.head())

#calculate descriptive stats for each study id
waso_stats = df_sleep_f5.groupby('study_id')['awakeDurationInMin'].agg(
    mean_waso='mean',
    median_waso='median',
    std_waso='std',
).reset_index()
display(waso_stats.head())

#create plots
fig_f5 = plot_descriptive_stats(
    df = df_sleep_f5,
    col_df = "awakeDurationInMin", 
    df_descriptive_stats=waso_stats,
    col_mean="mean_waso",
    col_median="median_waso", 
    n_patients=n_sleep_summary_raw,
    feature="Wake After Sleep Onset",
    title="Wake After Sleep Onset [min]",
    colors = COLORS
    )
fig_f5.show()


#add waso stats to feature dataframe
df_features = df_features.merge(
    waso_stats[['study_id', 'mean_waso', 'median_waso', 'std_waso']],
    on='study_id',
    how='outer'
)
display(df_features.head())

#add daily waso to df_features_daily
df_features_daily = df_features_daily.merge(
    df_sleep_f5[['study_id', 'calendarDate', 'awakeDurationInMin']],
    on=['study_id', 'calendarDate'],
    how='outer'
)
df_features_daily = df_features_daily.rename(columns={'awakeDurationInMin': 'daily_waso_in_min'})
display(df_features_daily.head())

### Counts

In [ ]:
df_sleep_f6 = df_sleep_stage_raw[['sleepSummaryId', 'datetime', 'datetime_utc', 'timezone', 'study_id','type']]

# Parse UTC once
dt_utc = pd.to_datetime(
    df_sleep_f6["datetime_utc"],
    errors="coerce",
    utc=True
)

# Convert each row to its own local timezone, then extract local date
df_sleep_f6["calendarDate"] = [
    ts.tz_convert(tz).date() if pd.notna(ts) and pd.notna(tz) else pd.NaT
    for ts, tz in zip(dt_utc, df_sleep_f6["timezone"])
]
display(df_sleep_f6.head())

#group by all columns and count occurrences of each sleep stage type
sleep_stage_counts = df_sleep_f6.groupby(['sleepSummaryId','study_id','type']).size().unstack(fill_value=0).reset_index()

#add date corresponding to sleepSummaryId using df_sleep_summary_raw and drop when no match is found
sleep_stage_counts = sleep_stage_counts.merge(
    df_sleep_summary_raw[['sleepSummaryId', 'calendarDate']],
    on='sleepSummaryId',
    how='left'
).dropna(subset=['calendarDate'])
display(sleep_stage_counts.head())

#calculate descriptive stats for each study id
awake_stats = sleep_stage_counts.groupby('study_id')['awake'].agg(
    mean_awake='mean',
    median_awake='median',
    std_awake='std',
).reset_index()
display(awake_stats.head())


awake_counts = sleep_stage_counts.groupby(['study_id', 'awake']).size().reset_index(name='nights')

display(awake_counts.head())

#create plots
fig_f6 = make_subplots(
    rows=3, cols=2, 
    specs=[
        [{"colspan": 2}, None],   # Row 1: spans both columns
        [{"colspan": 2}, None],   # Row 2: spans both columns
        [{}, {}]                   # Row 3: two separate columns
    ],
    subplot_titles=(
    "Number of Wake Ups during Night per Study ID",
    f"Distribution of Daily Wake After Sleep Onset Count",
    f"Distribution of Mean Wake After Sleep Onset Count, n = {n_sleep_summary_raw}",
    f"Distribution of Median Wake After Sleep Onset Count, n = {n_sleep_summary_raw}",
))

# --- Bubble scatter (row 1) ---
fig_f6.add_trace(
    go.Scatter(
        x=awake_counts['study_id'],
        y=awake_counts['awake'],
        mode='markers',
        marker=dict(size=awake_counts['nights'], sizemode='area', sizeref=2.*awake_counts['nights'].max()/(40**2), color=COLORS.get("blue1"), opacity=0.6),
        name='Awake Count',
        showlegend=False
    ),
    row=1, col=1
)

# Mean overlay
fig_f6.add_trace(
    go.Scatter(
        x=awake_stats['study_id'],
        y=awake_stats['mean_awake'],
        mode='markers',
        name='Mean Awake Count',
        marker=dict(color=COLORS.get("red1"), size=8)
    ),
    row=1, col=1
)

# Median overlay
fig_f6.add_trace(
    go.Scatter(
        x=awake_stats['study_id'],
        y=awake_stats['median_awake'],
        mode='markers',
        name='Median Awake Count',
        marker=dict(color=COLORS.get("orange1"), size=8, symbol='diamond')
    ),
    row=1, col=1
)

# --- Histogram (row 2) ---
fig_f6.add_trace(
    go.Histogram(
        x=awake_counts['awake'],
        xbins=dict(
            start=awake_counts['awake'].min() - 0.5,
            end=awake_counts['awake'].max() + 0.5,
            size=1
        ),
        marker_line_width=1,
        marker_line_color='black',
        marker_color=COLORS.get("blue1"),
        name='Daily WASO Count',
        showlegend=False
    ),
    row=2, col=1
)

fig_f6.add_trace(
    go.Histogram(
        x=awake_stats['mean_awake'],
        xbins=dict(
            start=0,
            end=awake_stats['mean_awake'].max() + 0.5,
            size=1
        ),
        marker_line_width=1,
        marker_line_color='black',
        marker_color=COLORS.get("blue1"),
        name='Mean WASO Count',
        showlegend=False
    ),
    row=3, col=1
)

fig_f6.add_trace(
    go.Histogram(
        x=awake_stats['median_awake'],
        xbins=dict(
            start=0,
            end=awake_stats['median_awake'].max() + 0.5,
            size=1
        ),
        marker_line_width=1,
        marker_line_color='black',
        marker_color=COLORS.get("blue1"),
        name='Median WASO Count',
        showlegend=False
    ),
    row=3, col=2
)




fig_f6.update_yaxes(title_text='Awake Count', dtick=1, row=1, col=1)
fig_f6.update_xaxes(title_text='Study ID', row=1, col=1)
fig_f6.update_xaxes(title_text='Count WASO', row=2, col=1)
fig_f6.update_yaxes(title_text='Count', row=2, col=1)
fig_f6.update_xaxes(title_text='Mean Count WASO', row=3, col=1)
fig_f6.update_yaxes(title_text='Count', row=3, col=1)
fig_f6.update_xaxes(title_text='Median Count WASO', row=3, col=2)
fig_f6.update_yaxes(title_text='Count', row=3, col=2)

fig_f6.update_layout(
    title_text="Wake After Sleep Onset Summary",
    height=1200
)

fig_f6.show()

#add awake count stats to feature dataframe
df_features = df_features.merge(
    awake_stats[['study_id', 'mean_awake', 'median_awake', 'std_awake']],
    on='study_id',
    how='outer'
)
display(df_features.head())

#add daily awake count to df_features_daily
df_features_daily = df_features_daily.merge(
    sleep_stage_counts[['study_id', 'calendarDate', 'awake']],
    on=['study_id', 'calendarDate'],
    how='outer'
)
df_features_daily = df_features_daily.rename(columns={'awake': 'daily_awake_count'})
display(df_features_daily.head())

## Sleep Efficiency Ratio SER

In [ ]:
df_sleep_f7 = df_sleep_summary_raw[['study_id', 'calendarDate', 'unmeasurableSleepInMs', 'deepSleepDurationInMs', 'lightSleepDurationInMs','remSleepInMs', 'awakeDurationInMs' ]].copy()

#calculate SER for each row: SER = light + deep + rem / light + deep + rem + awake *100
df_sleep_f7['SER'] = (df_sleep_f7['lightSleepDurationInMs'] + df_sleep_f7['deepSleepDurationInMs'] + df_sleep_f7['remSleepInMs']) / (df_sleep_f7['lightSleepDurationInMs'] + df_sleep_f7['deepSleepDurationInMs'] + df_sleep_f7['remSleepInMs'] + df_sleep_f7['awakeDurationInMs']) * 100
display(df_sleep_f7.head())

#calculate descriptive stats for each study id
ser_stats = df_sleep_f7.groupby('study_id')['SER'].agg(
    mean_ser='mean',
    median_ser='median',
    std_ser='std',
    cv_SER = lambda x: x.std() / x.mean() * 100
).reset_index()
display(ser_stats.head())

df_sleep_f7 = df_sleep_f7.sort_values(by=['study_id']).reset_index(drop=True)

#create plots
fig_f7 = plot_descriptive_stats(
    df = df_sleep_f7,
    col_df = "SER", 
    df_descriptive_stats=ser_stats,
    col_mean="mean_ser",
    col_median="median_ser", 
    n_patients=n_sleep_summary_raw,
    feature="Sleep Efficiency Ratio SER",
    title="SER [%]", 
    colors = COLORS
    )
fig_f7.show()

#add SER stats to feature dataframe
df_features = df_features.merge(
    ser_stats[['study_id', 'mean_ser', 'median_ser', 'std_ser']],
    on='study_id',
    how='outer'
)
display(df_features.head())

#add daily SER to df_features_daily
df_features_daily = df_features_daily.merge(
    df_sleep_f7[['study_id', 'calendarDate', 'SER']],
    on=['study_id', 'calendarDate'],
    how='outer'
)
df_features_daily = df_features_daily.rename(columns={'SER': 'daily_SER'})
display(df_features_daily.head())

## Store df_feature_sleep

In [ ]:
#if there already exists a file with same name move it to subfolder "archive"
if not os.path.exists('../../output/1_feature_extraction/archive'):
    os.makedirs('../../output/1_feature_extraction/archive')
files = os.listdir('../../output/1_feature_extraction')
for file in files:
    if file.startswith('df_features_sleep_') and file.endswith('.csv'):
        os.rename(f'../../output/1_feature_extraction/{file}', f'../../output/1_feature_extraction/archive/{file}')
    elif file.startswith('df_features_daily_sleep_') and file.endswith('.csv'):
        os.rename(f'../../output/1_feature_extraction/{file}', f'../../output/1_feature_extraction/archive/{file}')

#store df_feature as csv
date = datetime.now().strftime("%Y-%m-%d")
df_features.to_csv(f'../../output/1_feature_extraction/df_features_sleep_{date}.csv', index=False)
df_features_daily.to_csv(f'../../output/1_feature_extraction/df_features_daily_sleep_{date}.csv', index=False)